# TP4 : garanties de livraison et panne de broker

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 16:00-17:00, après la démo du TP (15:55-16:00) ; puis démonstration D4 par le formateur (17:00-17:10) |
| Compétences | C7 Paramètres de fiabilité ; C8 Panne de broker et réplication |
| Slides de rappel | S36 (ISR, LEO, HW), S37 (`min.insync.replicas` × `acks`), S38 (élection et ELR), S40 (panne contre perte de quorum) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

Mode conseillé : `extended`, pour voir les alertes à l'étape 6.7. En mode `core`, cette étape est sautée.

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez les trois cellules du début (connexion, fonctions d'aide, outils du TP), puis les cellules « Code fourni » des étapes déjà faites. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une cellule **`# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- **Vous n'avez pas de code à écrire.** Dans une cellule `# TODO`, remplacez chaque `...` par une valeur : un **réglage** (`"all"`, une option de commande...) ou votre **prévision** du résultat (un nombre, `True` ou `False`). Les noms qui commencent par `PREDICTED_` sont des prévisions : décidez-les avant d'exécuter, la vérification les compare au résultat.
- Sous les réponses, le code est fourni : lisez ses commentaires, ne le modifiez pas. Tant qu'il reste un `...`, la ligne `if filled(...):` l'empêche de s'exécuter et affiche « À compléter ».
- Ce TP arrête un nœud : les exemples en ligne de commande redirigent les erreurs (`2>/dev/null`), car `$BS` contient encore l'adresse du nœud arrêté et chaque commande afficherait `Couldn't resolve server kafka-3:19092`.
- Pendant la panne, des lignes commençant par `%3|` ou `%4|` peuvent apparaître sous une cellule : c'est le journal de librdkafka, la bibliothèque du client Python, qui signale ses tentatives de connexion au nœud arrêté. Ce ne sont pas des erreurs du notebook.
- Les fonctions `end_offsets()` et `read()` du TP2 sont fournies dans la cellule suivante.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
PROMETHEUS = "http://prometheus:9090"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Trois fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

def filled(*answers):
    """Vrai si toutes les réponses d'une cellule TODO sont données ; sinon affiche « À compléter »."""
    for answer in answers:
        if answer is ...:
            print("À compléter : remplacez chaque ... de la cellule par votre réponse, puis exécutez-la de nouveau.")
            return False
    return True

In [ ]:
# Outils du TP : imports, et deux fonctions du TP2
import threading
import time

import requests
from confluent_kafka import Consumer, ElectionType, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource, NewTopic, OffsetSpec


def end_offsets(topic):
    """{partition: offset de fin} : l'offset que recevra le prochain message de chaque partition."""
    admin = AdminClient(CONF)
    num_partitions = len(admin.list_topics(topic, timeout=10).topics[topic].partitions)
    # Une seule requête pour toutes les partitions ; la réponse contient un futur par partition
    futures = admin.list_offsets({TopicPartition(topic, p): OffsetSpec.latest() for p in range(num_partitions)})
    return {tp.partition: f.result().offset for tp, f in sorted(futures.items(), key=lambda x: x[0].partition)}


def read(topic, partition, start, count):
    """Lit au plus count messages de la partition, à partir de l'offset start."""
    consumer = Consumer({"bootstrap.servers": BS, "group.id": "tp4-lecture",
                         "enable.auto.commit": False, "log_level": 4})
    # assign() : lecture directe de la partition, sans groupe actif
    consumer.assign([TopicPartition(topic, partition, start)])
    messages = []
    while len(messages) < count:
        batch = consumer.consume(count - len(messages), timeout=10)  # liste vide si rien en 10 s
        if not batch:
            break
        messages += [m for m in batch if m.error() is None]
    consumer.close()
    return messages


print("Bootstrap INTERNAL :", BS)

## 1. Objectifs

À la fin du TP, vous savez :

- fixer le nombre minimal de répliques synchronisées au bon niveau (cluster, topic) ;
- prédire le résultat d'une production selon `acks` et `min.insync.replicas` quand un broker manque ;
- diagnostiquer une panne de broker : partitions sous-répliquées, ISR, ELR, quorum, brokers enregistrés ;
- vérifier qu'un producteur `acks=all` n'a perdu aucun message pendant la panne ;
- ramener le cluster à l'état nominal : rattrapage des répliques et élection des leaders préférés.

## 2. Prérequis

- TP1 à TP3 terminés, cluster sain : `scripts/validate.sh` au vert.
- Notions : slides S35 à S41.

## 3. Architecture utilisée

```
  Avant la panne                          Pendant la panne (kafka-3 arrêté)

  kafka-1   kafka-2   kafka-3             kafka-1   kafka-2   kafka-3
   [L]       [F]       [F]   payments-1    [L]       [F]        X
   [F]       [L]       [F]   payments-2    [F]       [L]        X
   [F]       [F]       [L]   payments-0    [L]       [F]        X   <- nouveau leader élu

  ISR = {1,2,3}                           ISR = {1,2}  (2 >= min.insync.replicas = 2 : écriture acceptée)
  refunds : min.insync.replicas = 3       ISR = {1,2}  (2 <  3 : écriture acks=all refusée)

  Quorum KRaft : 3 votants. Avec 2 votants sur 3, la majorité est conservée.
```

| Topic | Partitions | RF | `min.insync.replicas` |
|---|---|---|---|
| `payments` | 3 | 3 | 2 (hérité du cluster) |
| `refunds` | 1 | 3 | 3 (posé sur le topic) |

## 4. Contexte métier

Pour NovaShop, un paiement perdu est un incident grave : le client est débité et la commande n'est jamais expédiée. La direction exige que le flux `payments` résiste à la perte d'un nœud, sans perte et sans interruption. Pour les remboursements, l'équipe Finance a demandé une confirmation par les 3 répliques.

Vous devez configurer ces exigences, puis les prouver en arrêtant un nœud en pleine charge. Le bilan de l'exercice s'affiche en fin de notebook.

## 5. Énoncé

1. Exigez au moins 2 répliques synchronisées par défaut sur tout le cluster.
2. Créez `payments` et `refunds` selon le tableau de la section 4.
3. Arrêtez `kafka-3` pendant qu'un producteur `acks=all` écrit 100 paiements par seconde.
4. Diagnostiquez l'état du cluster, prédisez puis vérifiez le comportement des productions selon `acks`.
5. Redémarrez `kafka-3`, ramenez le cluster à l'état nominal et prouvez qu'aucun paiement n'a été perdu.

## 6. Étapes

### 6.1 Fixer le minimum ISR du cluster

`min.insync.replicas` fixe le nombre minimal de répliques synchronisées pour qu'une écriture `acks=all` soit acceptée. Depuis Kafka 4.1, l'ELR (Eligible Leader Replicas) est actif et impose une valeur unique pour tout le cluster, modifiable aussi par topic.

**Exemple en ligne de commande** : la configuration dynamique par défaut des brokers.

In [ ]:
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type brokers --entity-default

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers --entity-name 1 \
  --add-config min.insync.replicas=2
kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers --entity-default \
  --add-config min.insync.replicas=2
```

In [ ]:
# TODO 6.1 : remplacez les deux ... puis exécutez
PREDICTED_BROKER_OK = ...  # le réglage min.insync.replicas=2 sur le seul broker 1 sera-t-il accepté ?
                           # (True ou False)
ENTITY_OPTION = ...        # option de kafka-configs.sh qui vise la configuration de TOUS les brokers :
                           # "--entity-name 1" ou "--entity-default" ?

# Code fourni, à lire : rien à modifier en dessous
broker_refusal = None
if filled(PREDICTED_BROKER_OK, ENTITY_OPTION):
    admin = AdminClient(CONF)
    # 1. En Python, sur le seul broker 1 : le refus éventuel arrive dans une KafkaException
    resource = ConfigResource("broker", "1", incremental_configs=[
        ConfigEntry("min.insync.replicas", "2", incremental_operation=AlterConfigOpType.SET)])
    try:
        for future in admin.incremental_alter_configs([resource]).values():
            future.result()
    except KafkaException as e:
        broker_refusal = e.args[0].str()  # le message d'erreur du cluster

    # 2. Avec l'outil Java, sur l'entité choisie. Le client Python ne sait pas viser tous les brokers :
    #    ConfigResource("broker", "") lève ValueError.
    !docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers {ENTITY_OPTION} --add-config min.insync.replicas=2

print("refus_broker :", broker_refusal)

In [ ]:
# Vérification 6.1
assert broker_refusal is not None and "ELR" in broker_refusal, "À revoir : réglage par broker refusé : %s" % broker_refusal
print("OK  réglage par broker refusé : %s" % broker_refusal)
default = !docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type brokers --entity-default 2>&1
default = default.n
assert "min.insync.replicas=2 " in default, "À revoir : min.insync.replicas=2 pour tout le cluster"
print("OK  min.insync.replicas=2 pour tout le cluster")
assert PREDICTED_BROKER_OK is False, "À revoir : prévision %r : le réglage par broker est refusé quand l'ELR est actif" % (PREDICTED_BROKER_OK,)
print("OK  prévision juste : réglage par broker refusé")

**Résultat attendu** :

- valeur de départ : `min.insync.replicas=1`, source `DYNAMIC_DEFAULT_BROKER_CONFIG` ;
- le réglage sur le seul broker 1 est refusé : `Broker-level min.insync.replicas cannot be altered while ELR is enabled.` ;
- le réglage par défaut du cluster réussit : `Completed updating default config for brokers in the cluster.`

### 6.2 Créer `payments` et `refunds`

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic payments --partitions 3 --replication-factor 3
kafka-topics.sh --bootstrap-server $BS --create --topic refunds --partitions 1 --replication-factor 3 \
  --config min.insync.replicas=3
```

In [ ]:
# TODO 6.2 : remplacez le ... par votre prévision, puis exécutez UNE SEULE FOIS
PREDICTED_PAYMENTS_SOURCE = ...  # payments est créé sans min.insync.replicas. D'où vient sa valeur :
                                 # "DYNAMIC_TOPIC_CONFIG" (posée sur le topic)
                                 # ou "DYNAMIC_DEFAULT_BROKER_CONFIG" (réglage du cluster, étape 6.1) ?

# Code fourni, à lire : rien à modifier en dessous
min_isr = {}
if filled(PREDICTED_PAYMENTS_SOURCE):
    admin = AdminClient(CONF)
    # Les deux topics du tableau de la section 3 ; refunds porte sa propre valeur
    futures = admin.create_topics([NewTopic("payments", num_partitions=3, replication_factor=3),
                                   NewTopic("refunds", num_partitions=1, replication_factor=3,
                                            config={"min.insync.replicas": "3"})])
    for future in futures.values():
        future.result()

    # Valeur de min.insync.replicas de chaque topic, et son origine (source)
    for topic in ("payments", "refunds"):
        future = list(admin.describe_configs([ConfigResource("topic", topic)]).values())[0]
        entry = future.result()["min.insync.replicas"]
        min_isr[topic] = (entry.value, ConfigSource(entry.source).name)

print(min_isr)

In [ ]:
# Vérification 6.2
assert min_isr.get("payments") == ("2", "DYNAMIC_DEFAULT_BROKER_CONFIG"), "À revoir : payments hérité 2 du cluster"
print("OK  payments hérité 2 du cluster")
assert min_isr.get("refunds") == ("3", "DYNAMIC_TOPIC_CONFIG"), "À revoir : refunds exige 3 répliques"
print("OK  refunds exige 3 répliques")
assert PREDICTED_PAYMENTS_SOURCE == min_isr["payments"][1], "À revoir : prévision %r, origine observée %s" % (PREDICTED_PAYMENTS_SOURCE, min_isr["payments"][1])
print("OK  prévision juste : valeur héritée du cluster")

In [ ]:
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --topic 'payments|refunds'"

**Résultat attendu** : `payments` hérite `min.insync.replicas=2` du cluster (`DYNAMIC_DEFAULT_BROKER_CONFIG`) ; `refunds` a sa propre valeur, 3. Chaque partition a 3 répliques, toutes dans l'ISR.

### 6.3 Produire en `acks=all`

Avec `acks=all`, le leader ne confirme un message qu'une fois écrit par toutes les répliques de l'ISR, et seulement si l'ISR compte au moins `min.insync.replicas` répliques.

Équivalent en ligne de commande (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
seq 1 100 | sed 's/^/PAY-/' | kafka-console-producer.sh --bootstrap-server $BS --topic payments \
  --request-required-acks all
```

In [ ]:
# TODO 6.3 : remplacez le ... puis exécutez UNE SEULE FOIS
ACKS = ...  # niveau d'acquittement : "0", "1" ou "all" ? Le paiement ne doit être confirmé
            # qu'une fois copié sur toutes les répliques synchronisées.

# Code fourni, à lire : rien à modifier en dessous
admin = AdminClient(CONF)
confirmed_73 = 0
if filled(ACKS):
    def count_delivered(err, msg):
        """Rapport de livraison : compte les messages confirmés sans erreur."""
        global confirmed_73
        if err is None:
            confirmed_73 += 1

    producer_all = Producer({"bootstrap.servers": BS, "acks": ACKS, "log_level": 4})
    # PAY-1 à PAY-100, sans clé
    for i in range(1, 101):
        producer_all.produce("payments", value="PAY-%d" % i, on_delivery=count_delivered)
    producer_all.flush(30)  # attend toutes les confirmations, 30 s au plus

print(confirmed_73, "confirmés ;", end_offsets("payments") if "payments" in admin.list_topics(timeout=10).topics else "")

In [ ]:
# Vérification 6.3
assert confirmed_73 == 100, "À revoir : 100 paiements confirmés"
print("OK  100 paiements confirmés")
assert sum(end_offsets("payments").values()) == 100, "À revoir : 100 messages dans payments"
print("OK  100 messages dans payments")
assert ACKS == "all", "À revoir : acks=%r : seul all attend la copie sur les répliques synchronisées" % (ACKS,)
print("OK  acks=all")

**Résultat attendu** : 100 confirmations. Sans clé, les messages se concentrent dans une partition (partitionneur « sticky », TP2).

### 6.4 Charge continue et arrêt de `kafka-3`

La classe `Load` ci-dessous est un producteur qui tourne en arrière-plan : il envoie `count` messages à `rate` messages par seconde et compte les confirmations (`confirmed`) et les erreurs définitives (`errors`). Elle remplace `kafka-producer-perf-test.sh` de la version Markdown.

In [ ]:
# Code fourni : un producteur de charge qui tourne en arrière-plan
class Load:
    """Producteur de charge en arrière-plan : count messages, à rate messages par seconde."""

    def __init__(self, topic, count, rate, config):
        self.confirmed = 0  # messages confirmés par le cluster
        self.errors = []    # erreurs définitives (après toutes les nouvelles tentatives)
        self.producer = Producer({"bootstrap.servers": BS, "log_level": 4, **config})
        # L'envoi tourne dans un fil d'exécution à part : le notebook reste disponible
        self._thread = threading.Thread(target=self._send, args=(topic, count, rate), daemon=True)
        self._thread.start()

    def _report(self, err, msg):
        # Rapport de livraison de chaque message
        if err is None:
            self.confirmed += 1
        else:
            self.errors.append(err)

    def _send(self, topic, count, rate):
        start = time.time()
        for i in range(count):
            # Cadence régulière : le message i part à start + i / rate
            pause = start + i / rate - time.time()
            if pause > 0:
                time.sleep(pause)
            self.producer.produce(topic, value="CHARGE-%d" % (i + 1), on_delivery=self._report)
            self.producer.poll(0)  # traite les rapports de livraison déjà arrivés
        self.producer.flush(120)   # à la fin, attend les dernières confirmations

    def running(self):
        """Vrai tant que l'envoi n'est pas terminé."""
        return self._thread.is_alive()

    def __repr__(self):
        # Texte affiché par print(load)
        return "Charge(%d confirmés, %d erreurs%s)" % (self.confirmed, len(self.errors),
                                                      ", en cours" if self.running() else ", terminée")

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose exec -d tools bash -c 'kafka-producer-perf-test.sh --bootstrap-server $BS --topic payments \
  --num-records 6000 --throughput 100 --record-size 200 --command-property acks=all \
  > /tmp/perf-payments.log 2>&1'
sleep 5
time docker compose stop kafka-3
```

**Exécutez la cellule `TODO` une seule fois**, d'un seul bloc : le délai de 5 s entre la charge et l'arrêt compte.

In [ ]:
# TODO 6.4 : remplacez les deux ... puis exécutez UNE SEULE FOIS, d'un seul bloc
IDEMPOTENCE = ...            # enable.idempotence : True ou False ? Il empêche les doublons quand un envoi
                             # est rejoué après un changement de leader.
PREDICTED_LOAD_ERRORS = ...  # pendant l'arrêt contrôlé de kafka-3, le producteur de charge recevra-t-il
                             # des erreurs définitives ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
load = None
downtime = None
if filled(IDEMPOTENCE, PREDICTED_LOAD_ERRORS):
    # 6000 paiements à 100 par seconde, en acks=all : environ 60 s de charge
    load = Load("payments", 6000, 100, {"acks": "all", "enable.idempotence": IDEMPOTENCE})
    time.sleep(5)  # la charge tourne depuis 5 s
    start = time.time()
    !docker compose stop kafka-3
    downtime = round(time.time() - start, 1)  # durée de l'arrêt contrôlé

print(load, "- arrêt en", downtime, "s")

In [ ]:
# Vérification 6.4
assert load is not None and load.running(), "À revoir : charge en cours : %s" % load
print("OK  charge en cours : %s" % load)
output = !docker compose ps --format '{{{{.Name}}}}' kafka-3 2>&1
assert output.n == "", "À revoir : kafka-3 arrêté"
print("OK  kafka-3 arrêté")
assert PREDICTED_LOAD_ERRORS == bool(load.errors), "À revoir : prévision %r, %d erreur(s) définitive(s)" % (PREDICTED_LOAD_ERRORS, len(load.errors))
print("OK  prévision juste : aucune erreur définitive")

**Résultat attendu** : `docker compose stop kafka-3` dure environ 3 s. La charge continue, sans erreur.

### 6.5 Diagnostiquer la panne

**Exemples en ligne de commande** : partitions sous-répliquées, `payments`, partitions au minimum ISR et sous le minimum, quorum, brokers enregistrés.

In [ ]:
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --under-replicated-partitions 2>/dev/null | grep -c Partition"
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --topic payments 2>/dev/null"
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --at-min-isr-partitions 2>/dev/null | grep -c Partition"
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --under-min-isr-partitions 2>/dev/null"
!docker exec kafka-tools bash -c "kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication 2>/dev/null"
!docker exec kafka-tools bash -c "kafka-cluster.sh list-endpoints --bootstrap-server $BS 2>/dev/null"

**Code fourni** : `partition_state()` donne, pour chaque partition du cluster, son leader, ses répliques et son ISR. Elle resservira à l'étape 6.8.

In [ ]:
# Code fourni : état de toutes les partitions du cluster
def partition_state():
    """Liste de tuples (topic, partition, leader, répliques, isr), identifiants de nœuds."""
    admin = AdminClient(CONF)
    lines = []
    # list_topics() : métadonnées de tous les topics ; chaque topic a un dictionnaire de partitions
    for name, topic in admin.list_topics(timeout=10).topics.items():
        for p in topic.partitions.values():
            lines.append((name, p.id, p.leader, p.replicas, p.isrs))
    return lines

In [ ]:
# TODO 6.5 : remplacez les deux ... par vos prévisions, puis exécutez
PREDICTED_LIVE_BROKERS = ...   # combien de brokers sont enregistrés pendant la panne ? (un nombre)
PREDICTED_PAYMENTS_ISR = ...   # combien de répliques dans l'ISR de chaque partition de payments ? (un nombre)

# Code fourni, à lire : rien à modifier en dessous
urp = None
payments_state = []
live_brokers = []
if filled(PREDICTED_LIVE_BROKERS, PREDICTED_PAYMENTS_ISR):
    admin = AdminClient(CONF)
    # Partitions sous-répliquées : moins de répliques dans l'ISR que de répliques prévues
    urp = sum(1 for _, _, _, replicas, isr in partition_state() if len(isr) < len(replicas))
    payments_state = sorted(line for line in partition_state() if line[0] == "payments")
    # Brokers enregistrés auprès du contrôleur
    live_brokers = sorted(n.id for n in admin.describe_cluster().result().nodes)

print("URP :", urp, "- brokers :", live_brokers)
table(payments_state, ["topic", "partition", "leader", "répliques", "isr"])

In [ ]:
# Vérification 6.5
assert urp is not None and urp >= 50, "À revoir : partitions sous-répliquées : %s (dont les 50 de __consumer_offsets)" % urp
print("OK  partitions sous-répliquées : %s (dont les 50 de __consumer_offsets)" % urp)
assert len(payments_state) == 3 and all(l[2] != 3 and 3 not in l[4] for l in payments_state), "À revoir : payments : aucun leader ni aucune réplique synchronisée sur kafka-3"
print("OK  payments : aucun leader ni aucune réplique synchronisée sur kafka-3")
assert live_brokers == [1, 2], "À revoir : brokers enregistrés : %s" % live_brokers
print("OK  brokers enregistrés : %s" % live_brokers)
assert PREDICTED_LIVE_BROKERS == len(live_brokers), "À revoir : prévision %r, %d brokers enregistrés" % (PREDICTED_LIVE_BROKERS, len(live_brokers))
print("OK  prévision juste : %d brokers" % len(live_brokers))
assert PREDICTED_PAYMENTS_ISR == len(payments_state[0][4]), "À revoir : prévision %r, ISR de %d" % (PREDICTED_PAYMENTS_ISR, len(payments_state[0][4]))
print("OK  prévision juste : ISR de %d" % len(payments_state[0][4]))

**Résultat attendu** :

| Mesure | Résultat |
|---|---|
| Partitions sous-répliquées | plusieurs dizaines : toutes celles qui avaient une réplique sur `kafka-3` |
| `payments` | aucun leader sur 3 ; ISR `[1, 2]` ; liste des répliques inchangée |
| Au minimum ISR | presque autant que les sous-répliquées |
| Sous le minimum ISR | `refunds`, avec `Isr: 2,1` (ou `1,2`) et `Elr: 3` |
| Quorum | `kafka-3` toujours listé comme `Follower`, avec un `Lag` qui augmente ; un leader existe |
| Brokers enregistrés | 1 et 2 seulement |

### 6.6 Prédire, puis produire

Trois productions vont être tentées pendant la panne. Prévoyez d'abord, pour chacune, si elle sera acceptée (`True`) ou refusée (`False`), **puis** exécutez la cellule `TODO`.

**Code fourni** : `produce()` envoie des valeurs avec le niveau d'acquittement demandé, et renvoie un résultat par message.

In [ ]:
# Code fourni : envoyer des valeurs avec un niveau d'acquittement donné
def produce(topic, values, acks):
    """Renvoie la liste des résultats, un par message : None si le message est écrit, sinon le code d'erreur."""
    results = []
    # retries=3 : trois nouvelles tentatives, comme le producteur console de Java. Sans cette limite,
    # librdkafka réessaie jusqu'à message.timeout.ms et l'erreur finale masque la cause.
    producer = Producer({"bootstrap.servers": BS, "acks": acks, "retries": 3,
                         "message.timeout.ms": 15000, "log_level": 4})
    for value in values:
        producer.produce(topic, value=value,
                         on_delivery=lambda err, msg: results.append(None if err is None else err.name()))
    producer.flush(30)
    return results

Équivalent en ligne de commande (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
seq 101 110 | sed 's/^/PAY-/' | kafka-console-producer.sh --bootstrap-server $BS --topic payments \
  --request-required-acks all
echo REF-1 | kafka-console-producer.sh --bootstrap-server $BS --topic refunds --request-required-acks all \
  --command-property delivery.timeout.ms=15000 --command-property request.timeout.ms=5000
echo REF-2 | kafka-console-producer.sh --bootstrap-server $BS --topic refunds --request-required-acks 1
```

In [ ]:
# TODO 6.6 : remplacez les trois ... par vos prévisions (True : acceptée, False : refusée), puis exécutez
PREDICTED_PAYMENTS_ALL = ...  # payments (minimum ISR 2, ISR de 2), 10 messages en acks=all
PREDICTED_REFUNDS_ALL = ...   # refunds (minimum ISR 3, ISR de 2), REF-1 en acks=all
PREDICTED_REFUNDS_1 = ...     # refunds (minimum ISR 3, ISR de 2), REF-2 en acks=1

# Code fourni, à lire : rien à modifier en dessous
results = {}
if filled(PREDICTED_PAYMENTS_ALL, PREDICTED_REFUNDS_ALL, PREDICTED_REFUNDS_1):
    results = {
        "payments, 10 messages, acks=all": produce("payments", ["PAY-%d" % i for i in range(101, 111)], "all"),
        "refunds, REF-1, acks=all": produce("refunds", ["REF-1"], "all"),
        "refunds, REF-2, acks=1": produce("refunds", ["REF-2"], 1),
    }
    predictions = [PREDICTED_PAYMENTS_ALL, PREDICTED_REFUNDS_ALL, PREDICTED_REFUNDS_1]
    # Une ligne par production : votre prévision, puis le résultat de chaque message
    table([(name, "acceptée" if p else "refusée", r) for (name, r), p in zip(results.items(), predictions)],
          ["Production", "Prévision", "Résultat"])

In [ ]:
# Vérification 6.6
assert results.get("payments, 10 messages, acks=all") == [None] * 10, "À revoir : payments en acks=all : accepté"
print("OK  payments en acks=all : accepté")
assert results["refunds, REF-1, acks=all"] == ["NOT_ENOUGH_REPLICAS"], "À revoir : refunds en acks=all : refusé"
print("OK  refunds en acks=all : refusé")
assert results["refunds, REF-2, acks=1"] == [None], "À revoir : refunds en acks=1 : accepté"
print("OK  refunds en acks=1 : accepté")
for (name, r), p in zip(results.items(), [PREDICTED_PAYMENTS_ALL, PREDICTED_REFUNDS_ALL, PREDICTED_REFUNDS_1]):
    accepted = all(x is None for x in r)
    assert p == accepted, "À revoir : %s : prévision %r, production %s" % (name, p, "acceptée" if accepted else "refusée")
print("OK  prévisions justes pour les trois productions")

**Résultat attendu** :

| Production | Résultat | Raison |
|---|---|---|
| `payments`, `acks=all` | Acceptée | ISR = 2 ≥ minimum 2 |
| `refunds`, `acks=all` | Refusée, `NOT_ENOUGH_REPLICAS` | ISR = 2 < minimum 3 |
| `refunds`, `acks=1` | Acceptée | Le minimum ISR ne s'applique qu'à `acks=all` |

### 6.7 Alertes

Prometheus évalue les règles d'alerte du lab toutes les 15 s. Une alerte passe par l'état `pending` pendant son délai `for`, puis `firing`. Interface : http://localhost:9090, menu « Alerts ».

**Exemple en ligne de commande** : l'API de Prometheus, depuis le conteneur `tools` (la version Markdown utilise `curl` sur l'hôte, avec `localhost:9090`). Juste après l'arrêt, la liste peut encore être vide : les alertes passent d'abord par `pending`.

In [ ]:
!docker exec kafka-tools bash -c 'wget -qO- http://prometheus:9090/api/v1/alerts | grep -o '"'"'"alertname":"[A-Za-z]*"'"'"' | sort | uniq -c || echo '"'"'Aucune alerte pour le moment, ou Prometheus absent (mode core)'"'"''

**Code fourni** : `active_alerts()` interroge l'API de Prometheus et renvoie le nom des alertes à l'état `firing`.

In [ ]:
# Code fourni : alertes actives dans Prometheus
def active_alerts():
    """Ensemble des noms des alertes à l'état firing."""
    alerts = requests.get(PROMETHEUS + "/api/v1/alerts", timeout=5).json()["data"]["alerts"]
    # Chaque alerte a un état (pending ou firing) et des étiquettes, dont son nom (alertname)
    return {a["labels"]["alertname"] for a in alerts if a["state"] == "firing"}

In [ ]:
# TODO 6.7 : remplacez le ... par votre réponse, puis exécutez (jusqu'à 2 minutes ; en mode core, passez)
PREDICTED_FIRST_STATE = ...  # avant de passer à "firing", dans quel état une alerte attend-elle
                             # la fin de son délai for : "inactive" ou "pending" ?

# Code fourni, à lire : rien à modifier en dessous
alerts = set()
if filled(PREDICTED_FIRST_STATE):
    try:
        requests.get(PROMETHEUS + "/-/ready", timeout=3)  # Prometheus est-il démarré (mode extended) ?
    except requests.RequestException:
        print("Prometheus absent (mode core) : étape 6.7 sautée")
    else:
        # Attente de KafkaNodeDown (délai for de 30 s), puis des alertes de délai for de 1 min
        wait_for(lambda: "KafkaNodeDown" in active_alerts(), timeout=150, step=5, message="alerte KafkaNodeDown")
        wait_for(lambda: len(active_alerts()) >= 4, timeout=90, step=5, message="4 alertes actives")
        alerts = active_alerts()

print(sorted(alerts))

In [ ]:
# Vérification 6.7
try:
    requests.get(PROMETHEUS + "/-/ready", timeout=3)
    assert "KafkaNodeDown" in alerts, "À revoir : alerte KafkaNodeDown active"
    print("OK  alerte KafkaNodeDown active")
    assert {"KafkaUnderReplicatedPartitions", "KafkaUnderMinIsrPartitions"} <= alerts, "À revoir : alertes de réplication actives : %s" % sorted(alerts)
    print("OK  alertes de réplication actives : %s" % sorted(alerts))
except requests.RequestException:
    print("Prometheus absent (mode core) : étape 6.7 sautée")
assert PREDICTED_FIRST_STATE == "pending", "À revoir : réponse %r : une alerte attend son délai for à l'état pending" % (PREDICTED_FIRST_STATE,)
print("OK  réponse juste : pending, puis firing")

**Résultat attendu** : environ une minute après l'arrêt, 4 alertes `firing` : `KafkaNodeDown`, `KafkaUnderMinIsrPartitions`, `KafkaUnderReplicatedPartitions` et `KafkaFencedBrokers`.

### 6.8 Redémarrer `kafka-3` et revenir à l'état nominal

Au redémarrage, les répliques de `kafka-3` rattrapent leur retard puis rentrent dans l'ISR. Mais `kafka-3` ne redevient leader de rien : il faut une **élection des leaders préférés** (le premier broker de chaque liste de répliques).

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose start kafka-3
# [tools]
kafka-leader-election.sh --bootstrap-server $BS --election-type preferred --all-topic-partitions
```

In [ ]:
# TODO 6.8 : remplacez les deux ... puis exécutez UNE SEULE FOIS (une minute environ)
PREDICTED_KAFKA3_LEADS = ...  # une fois revenu dans l'ISR, et AVANT l'élection, kafka-3 est-il de nouveau
                              # leader d'une partition de payments ? (True ou False)
ELECTION_TYPE = ...           # élection qui rend leur rôle aux leaders préférés :
                              # ElectionType.PREFERRED ou ElectionType.UNCLEAN ?

# Code fourni, à lire : rien à modifier en dessous
urp_duration = None
before_election = after_election = []
elected_count = None
if filled(PREDICTED_KAFKA3_LEADS, ELECTION_TYPE):
    admin = AdminClient(CONF)
    start = time.time()
    !docker compose start kafka-3
    # Attente : toutes les répliques sont revenues dans l'ISR (plus aucune partition sous-répliquée)
    wait_for(lambda: all(len(l[4]) == len(l[3]) for l in partition_state()), timeout=120,
             message="retour de toutes les répliques dans l'ISR")
    urp_duration = round(time.time() - start, 1)

    def payments_leaders():
        """(partition, leader, leader préféré) pour payments ; le préféré est la première réplique."""
        return sorted((l[1], l[2], l[3][0]) for l in partition_state() if l[0] == "payments")

    before_election = payments_leaders()
    # Élection sur toutes les partitions (None) ; le résultat a un élément par partition traitée
    elected_count = len(admin.elect_leaders(ELECTION_TYPE, None).result())
    after_election = payments_leaders()

print("URP à 0 en", urp_duration, "s ;", elected_count, "partitions élues")
table(before_election, ["partition", "leader", "préféré"])
table(after_election, ["partition", "leader", "préféré"])

In [ ]:
# Vérification 6.8
assert partition_state() and all(len(l[4]) == len(l[3]) for l in partition_state()), "À revoir : plus aucune partition sous-répliquée"
print("OK  plus aucune partition sous-répliquée")
assert any(leader != preferred for _, leader, preferred in before_election), "À revoir : avant l'élection, un leader de payments n'est pas le préféré"
print("OK  avant l'élection, un leader de payments n'est pas le préféré")
assert after_election and all(leader == preferred for _, leader, preferred in after_election), "À revoir : après l'élection, chaque partition de payments a son leader préféré"
print("OK  après l'élection, chaque partition de payments a son leader préféré")
kafka3_leads = any(leader == 3 for _, leader, _ in before_election)
assert PREDICTED_KAFKA3_LEADS == kafka3_leads, "À revoir : prévision %r : avant l'élection, kafka-3 n'est leader d'aucune partition de payments" % (PREDICTED_KAFKA3_LEADS,)
print("OK  prévision juste : kafka-3 sans leader avant l'élection")

In [ ]:
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --topic 'payments|refunds'"

**Résultat attendu** :

- plus aucune partition sous-répliquée quelques secondes après le démarrage ;
- avant l'élection, la partition de `payments` dont le leader préféré est 3 garde son leader de secours ;
- l'élection concerne plusieurs dizaines de partitions, puis chaque partition de `payments` retrouve son leader préféré.

### 6.9 Défi : prouver qu'aucun paiement n'est perdu

Combien de messages `payments` devrait-il contenir ? Prouvez-le avec deux sources indépendantes : les offsets du topic et le bilan du producteur de charge. Vérifiez aussi le contenu de `refunds`.

In [ ]:
# TODO 6.9 (défi) : remplacez les deux ... par vos prévisions, puis exécutez (jusqu'à 2 minutes)
PREDICTED_EXPECTED = ...  # combien de messages payments doit-il contenir ? Additionnez les envois
                          # confirmés des étapes 6.3, 6.4 et 6.6 (un nombre)
PREDICTED_REFUNDS = ...   # quelles valeurs le topic refunds contient-il ? Une liste de textes,
                          # par exemple ["REF-1", "REF-2"]

# Code fourni, à lire : rien à modifier en dessous
expected = total_payments = None
refunds_content = []
if filled(PREDICTED_EXPECTED, PREDICTED_REFUNDS):
    wait_for(lambda: not load.running(), timeout=120, message="fin de la charge")
    # Source 1 : ce que les producteurs ont vu confirmé (6.3, charge de 6.4, payments de 6.6)
    expected = confirmed_73 + load.confirmed + results["payments, 10 messages, acks=all"].count(None)
    # Source 2 : ce que le topic contient, somme des offsets de fin
    total_payments = sum(end_offsets("payments").values())
    # Toutes les valeurs de la partition 0 de refunds
    refunds_content = [m.value().decode() for m in read("refunds", 0, 0, end_offsets("refunds")[0])]

print("attendu :", expected, "- dans le topic :", total_payments, "- producteur de charge :", load)
print("refunds :", refunds_content)

In [ ]:
# Vérification 6.9
assert load is not None and load.confirmed == 6000 and not load.errors, "À revoir : 6000 envois de charge confirmés, aucune erreur définitive"
print("OK  6000 envois de charge confirmés, aucune erreur définitive")
assert expected == 6110 and total_payments == expected, "À revoir : " + ("payments contient %s messages pour %s attendus"
         % (total_payments, expected))
print("OK  " + ("payments contient %s messages pour %s attendus"
         % (total_payments, expected)))
assert refunds_content == ["REF-2"], "À revoir : refunds ne contient que REF-2"
print("OK  refunds ne contient que REF-2")
assert PREDICTED_EXPECTED == total_payments, "À revoir : prévision %r, %s messages dans payments" % (PREDICTED_EXPECTED, total_payments)
print("OK  prévision juste : %s messages" % total_payments)
assert PREDICTED_REFUNDS == refunds_content, "À revoir : prévision %r, refunds contient %s" % (PREDICTED_REFUNDS, refunds_content)
print("OK  prévision juste : refunds contient %s" % refunds_content)

**Résultat attendu** : `payments` contient 100 + 6000 + 10 = 6110 messages, autant que les confirmations reçues. Le producteur de charge n'a aucune erreur définitive. `refunds` contient seulement `REF-2`.

## Bilan de l'exercice

In [ ]:
# Code fourni : bilan de l'exercice, une ligne par contrôle ; « à compléter » si l'étape n'a pas été faite
output = !docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type brokers --entity-default 2>&1
table([
    ("min.insync.replicas du cluster", "2" if "min.insync.replicas=2 " in output.n else "à revoir", "6.1"),
    ("Arrêt de kafka-3", "%s s" % downtime if downtime else "à compléter", "6.4"),
    ("Partitions sous-répliquées pendant la panne", urp if urp is not None else "à compléter", "6.5"),
    # OK si tous les messages de la production sont écrits, sinon les codes d'erreur
    ("Productions (payments all, refunds all, refunds 1)",
     " / ".join("OK" if r and all(x is None for x in r) else str(r) for r in results.values()), "6.6"),
    ("Alertes actives", ", ".join(sorted(alerts)) or "aucune relevée", "6.7"),
    ("Retour de l'ISR après redémarrage", "%s s" % urp_duration if urp_duration else "à compléter", "6.8"),
    ("Messages dans payments / attendus", "%s / %s" % (total_payments, expected), "6.9"),
], ["Contrôle", "Valeur", "Étape"])

## 7. Questions de réflexion

1. Pourquoi le minimum ISR ne peut-il pas être fixé broker par broker quand l'ELR est actif ?
2. Avec RF 3 et `min.insync.replicas=2`, combien de brokers peut-on perdre sans bloquer les producteurs `acks=all` ? Et sans perdre de données ?
3. `refunds` avec `min.insync.replicas=3` : quel est le prix de cette exigence ? Que conseilleriez-vous à l'équipe Finance ?
4. Le message `REF-2` écrit en `acks=1` pendant la panne est-il en sécurité ? Pourquoi ?
5. Que représente `Elr: 3` sur `refunds` ?
6. Pourquoi faut-il une élection des leaders préférés après le retour d'un broker ? Que se passe-t-il si on ne la lance pas ?
7. Quelle différence entre perdre un broker (ce TP) et perdre le quorum (démonstration D4) ?